In [1]:
import os 
import certifi 
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor     #react agent is a type of agent that uses the ReAct framework, which stands for Reasoning and Acting. It allows the agent to reason about its actions and make decisions based on the information it has.

In [76]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=4)

In [5]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Autonomous AI systems test governance in physical environments\n\nAI in Action\n\nMay 26, 2026\n\n# Autonomous AI systems test governance in physical environments\n\nOpenAI opens Singapore AI lab as IMDA updates AI framework\n\nGovernance, Regulation & Policy\n\nMay 22, 2026\n\n# OpenAI opens Singapore AI lab as IMDA updates AI framework\n\nEnvironment & Sustainability\n\nMay 22, 2026\n\n# China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nArtificial Intelligence\n\nMay 22, 2026\n\n# Musk and Zuckerberg convinced Trump to scrap AI executive order\n\nAI in Action\n\nMay 21, 2026\n\n# Nvidia’s Vera chip is the US$200 billion bet Jensen Huang doesn’t want you to overlook\n\nAI in Action\n\nMay 20, 2026 [...] Artificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n### Quantum A

CURRENT WEATHER INFORMATION

In [6]:

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [7]:
print(type(get_weather_data))

<class 'langchain_core.tools.StructuredTool'>


In [8]:
tools = [
    TavilySearchResults(max_results=4),
    get_weather_data
]

In [9]:
tools

[TavilySearchResults(max_results=4),
 StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x0000017B3E13C860>)]

In [10]:
print(tools)

for t in tools:
    print(type(t))
    print(t.name)

[TavilySearchResults(max_results=4), StructuredTool(name='get_weather_data', description='get_weather_data(city: str) -> str - Fetch current weather information for a city.', args_schema=<class 'pydantic.v1.main.get_weather_dataSchema'>, func=<function get_weather_data at 0x0000017B3E13C860>)]
<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json
<class 'langchain_core.tools.StructuredTool'>
get_weather_data


In [80]:
# ==========================================
# LLM
# ==========================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [81]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI break up with its robot girlfriend? Because she had too many bugs in her system!', response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 13, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-dfef86ee-e10c-4764-885a-a75c33ef79a3-0')

In [11]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    max_tokens=500
)

In [12]:
response = llm.invoke("who won 2019 cricket world cup?")
response

AIMessage(content="The 2019 Cricket World Cup was won by England. They claimed their first World Cup title by defeating New Zealand in the final, which took place at Lord's in London on July 14, 2019. The match ended in a tie, and England won based on the boundary count rule after a Super Over also ended in a tie.", response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 16, 'total_tokens': 87, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 4.5e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 4.5e-05, 'upstream_inference_prompt_cost': 2.4e-06, 'upstream_inference_completions_cost': 4.26e-05}}, 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_eb37e061ec', 'finish_reason': 'stop', 'logprobs': None}, id='ru

In [13]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")           #why I use this prompt? because it is a prompt that is designed to work with the ReAct framework, which allows the agent to reason about its actions and make decisions based on the information it has. This prompt provides a structure for the agent to follow, which can help it to generate more coherent and relevant responses.

c:\Users\Gouthum\Downloads\Complete Agentic AI Coursework\5.Single AI Agent system using Langchain\venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [14]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [15]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [16]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [18]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(                      #AgentExecutor is a class that takes an agent and a list of tools, and allows you to execute the agent with the tools. It handles the interaction between the agent and the tools, allowing the agent to call the tools as needed to complete its tasks.
    agent=agent,                                     #agent=agent means that we are passing the agent we just created to the AgentExecutor, so that it can use that agent to execute tasks. we use only single agent in this course, so we can just pass that agent to the AgentExecutor.
    tools=tools,                                     #tools=tools means that we are passing the list of tools we created to the AgentExecutor, so that it can use those tools to execute tasks. the agent will be able to call these tools as needed to complete its tasks.
    verbose=True                                     #verbose=True means that we want the AgentExecutor to print out the steps it is taking as it executes tasks. This can be helpful for debugging and understanding how the agent is using the tools to complete its tasks.
)

In [19]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({                   #agent_executor.invoke is a method that takes an input and executes the agent with that input. The agent will use the tools as needed to complete the task specified in the input. The response will be the final output of the agent after it has completed its reasoning and actions based on the input.
    "input": (
        "Find the capital of Karnataka"
        "and then find its current weather."
    )
})




> Entering new AgentExecutor chain...
I need to first find out what the capital of Karnataka is. After that, I can check the current weather for that city. 

Action: tavily_search_results_json  
Action Input: "capital of Karnataka"  [{'url': 'https://www.vedantu.com/question-answer/the-capital-city-of-karnataka-is-achennai-class-9-social-science-cbse-5fc8d9015bf56b2541999712', 'content': 'The etymology of Karnataka is derived from the words ‘karu’ and ‘nadu’ which means ‘elevated land’. Karnataka has been the state for some of the most powerful empires of medieval India. Karnataka contributed massively towards the development of Hindustani Classical Music called Carnatic music. Bangalore is the capital city of Karnataka and has a population of 11 million people. It ranks third in the category of most populated city of India. Bangalore is popularly referred to as ‘Silicon valley of India’. It plays a leading role in the country’s growth in the sector of information technology. Bangalo

In [20]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({                   #agent_executor.invoke is a method that takes an input and executes the agent with that input. The agent will use the tools as needed to complete the task specified in the input. The response will be the final output of the agent after it has completed its reasoning and actions based on the input.
    "input": (
        "Tell me which is the next 3 international series of india in 2026"
        "and then find its current weather."
    )
})




> Entering new AgentExecutor chain...
I need to find out the next three international series that India will play in 2026. After that, I will check the current weather in the relevant cities where these series will take place. 

Action: tavily_search_results_json  
Action Input: "India international cricket series schedule 2026"  [{'url': 'https://m.economictimes.com/news/sports/cricket-calendar-2026-heres-full-schedule-of-team-india/articleshow/126283748.cms', 'content': 'Month | Date | Match | Venue || January | 11 Jan 2026 | India vs New Zealand, 1st ODI | BCA Stadium, Kotambi, Vadodara |\n| January | 14 Jan 2026 | India vs New Zealand, 2nd ODI | Niranjan Shah Stadium, Khandheri, Rajkot |\n| January | 18 Jan 2026 | India vs New Zealand, 3rd ODI | Holkar Cricket Stadium, Indore |\n| January | 21 Jan 2026 | India vs New Zealand, 1st T20I | Vidarbha Cricket Association Stadium, Nagpur |\n| January | 23 Jan 2026 | India vs New Zealand, 2nd T20I | Shaheed Veer Narayan Singh Internation

In [21]:
print(response["output"])

The next three international series of India in 2026 are:
1. **India vs New Zealand** (January 11-31, 2026) - Various cities in India.
2. **ICC Men's T20 World Cup 2026** (February 7 - March 8, 2026) - Mumbai.
3. **India vs Afghanistan** (June 6-20, 2026) - Various cities in India.

Current weather:
- Mumbai: 30°C, Clear, 69% humidity.
- Vadodara: 31°C, Patchy rain nearby, 57% humidity.
- Rajkot: 29°C, Clear, 65% humidity.
- Indore: 29°C, Clear, 48% humidity.
